<div class="alert alert-block alert-success">
  <h2>Hematology RAG Evaluation</h2>
</div>

<div class="alert alert-block alert-info">
    <h2>Import Libraries and Load Data</h2>
</div>

In [11]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

_cwd = Path.cwd()
project_root = _cwd if (_cwd / "dataset").exists() else _cwd.parent

In [12]:
try:
    import litellm
except ImportError:
    litellm = None

def load_rag_examples(golden_dir, n_positive=10, n_negative=10):
    # Load golden examples from hematology_golden
    positive_dir = golden_dir / "positive"
    negative_dir = golden_dir / "negative"
    positive_texts = []
    negative_texts = []
    if positive_dir.exists():
        for f in sorted(positive_dir.glob("*.txt"))[:n_positive]:
            positive_texts.append(f.read_text(encoding="utf-8", errors="replace").strip())
    if negative_dir.exists():
        for f in sorted(negative_dir.glob("*.txt"))[:n_negative]:
            negative_texts.append(f.read_text(encoding="utf-8", errors="replace").strip())
    return positive_texts, negative_texts

def _parse_verdict(text):
    if not text:
        return None
    for word in ("true", "false", "uncertain"):
        if word in text.split() or text.startswith(word):
            return word
    return None

def rag_classify(report_text, positive_examples, negative_examples, model_id="ollama/phi3:latest"):
    # Classify a report using RAG (golden examples + LLM)
    def block(label, texts):
        if not texts:
            return ""
        parts = [f"--- {label} ---"]
        for i, t in enumerate(texts, 1):
            parts.append(f"[Example {i}]\n{t}")
        return "\n\n".join(parts)

    positive_block = block("Golden examples labeled POSITIVE (pneumonia)", positive_examples)
    negative_block = block("Golden examples labeled NEGATIVE (no pneumonia)", negative_examples)
    current = (report_text or "").strip()

    prompt = f"""You are classifying a hematology report for pneumonia. Use the golden examples below as reference.

{positive_block}

{negative_block}

--- Report to classify ---
{current}

Based on the golden examples, is this report more like POSITIVE (pneumonia) or NEGATIVE (no pneumonia)? Reply with exactly one word: true, false, or uncertain."""

    if litellm is None:
        return "uncertain"
    try:
        out = litellm.completion(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20,
            temperature=0,
        )
        text = (out.choices[0].message.content or "").strip().lower()
        verdict = _parse_verdict(text)
        return verdict if verdict in ("true", "false", "uncertain") else "uncertain"
    except Exception:
        return "uncertain"

In [13]:
# Load golden examples and test set
golden_dir = project_root / "dataset" / "hematology_golden"
reports_dir = project_root / "dataset" / "heamatology_reports"
TEST_PATIENT_RANGE = range(11, 31)  # 20 per class, excludes golden set (1-10)

positive_examples, negative_examples = load_rag_examples(golden_dir)
print(f"RAG: {len(positive_examples)} positive examples, {len(negative_examples)} negative examples")

def load_test_set():
    samples = []
    for label, subdir in [("positive", "positive"), ("negative", "negative")]:
        d = reports_dir / subdir
        for i in TEST_PATIENT_RANGE:
            f = d / f"patient{i}.txt"
            if f.exists():
                text = f.read_text(encoding="utf-8", errors="replace").strip()
                samples.append({"path": str(f), "text": text, "label": label})
    return pd.DataFrame(samples)

df = load_test_set()
print(f"Test set: {len(df)} reports ({df['label'].value_counts().to_dict()})")
df.head()

RAG: 10 positive examples, 10 negative examples
Test set: 40 reports ({'positive': 20, 'negative': 20})


,path,text,label
0,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,Patient ID: BLOOD-PN-015\nAge / Sex: 84 / Fema...,positive
1,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,Patient ID: BLOOD-PN-015\nAge / Sex: 84 / Fema...,positive
2,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,Patient ID: BLOOD-PN-016\nAge / Sex: 59 / Male...,positive
3,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,Patient ID: BLOOD-PN-017\nAge / Sex: 61 / Fema...,positive
4,/Users/mathuria/Desktop/gitCode/MSDS498_Capsto...,Patient ID: BLOOD-PN-018\nAge / Sex: 48 / Male...,positive


In [ ]:
# Run RAG on each test report
model = os.environ.get("OLLAMA_MODEL", "phi3:latest")
model_id = f"ollama/{model}"

results = []
for _, row in df.iterrows():
    verdict = rag_classify(row["text"], positive_examples, negative_examples, model_id=model_id)
    results.append({
        "label": row["label"],
        "y_true": 1 if row["label"] == "positive" else 0,
        "verdict": verdict,
        "pred": 1 if verdict == "true" else (0 if verdict == "false" else None),
    })

results_df = pd.DataFrame(results)
results_df

,label,y_true,verdict,pred
0,positive,1,true,1.0
1,positive,1,true,1.0
2,positive,1,false,0.0
3,positive,1,true,1.0
4,positive,1,true,1.0
5,positive,1,true,1.0
6,positive,1,true,1.0
7,positive,1,true,1.0
8,positive,1,true,1.0
9,positive,1,true,1.0


<div class="alert alert-block alert-info">
    <h2>Compute RAG metrics</h2>
</div>

In [15]:
def compute_metrics(rdf, pred_col="pred"):
    # Compute accuracy, precision, recall, F1
    resolved = rdf[pred_col].notna()
    uncertain_rate = 1 - resolved.mean()
    strict_acc = (rdf[pred_col].notna() & (rdf[pred_col] == rdf["y_true"])).mean()

    if resolved.sum() > 0:
        y_t = rdf.loc[resolved, "y_true"].values
        y_p = rdf.loc[resolved, pred_col].astype(int).values
        prec, rec, f1, _ = precision_recall_fscore_support(y_t, y_p, average="binary", zero_division=0)
        resolved_acc = accuracy_score(y_t, y_p)
    else:
        prec = rec = f1 = resolved_acc = 0.0

    return {
        "strict_accuracy": strict_acc,
        "resolved_accuracy": resolved_acc if resolved.sum() > 0 else None,
        "uncertain_rate": uncertain_rate,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "n_resolved": int(resolved.sum()),
        "n_total": len(rdf),
    }

metrics = compute_metrics(results_df)
print("=== RAG Performance ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

=== RAG Performance ===
  strict_accuracy: 0.825
  resolved_accuracy: 0.9705882352941176
  uncertain_rate: 0.15000000000000002
  precision: 1.0
  recall: 0.9411764705882353
  f1: 0.9696969696969697
  n_resolved: 34
  n_total: 40


In [16]:
# Confusion matrix (negative, positive, uncertain)
labels = ["negative", "positive", "uncertain"]
y_p = []
for p in results_df["pred"]:
    if pd.isna(p) or p not in (0, 1):
        y_p.append(2)
    else:
        y_p.append(int(p))
cm = confusion_matrix(results_df["y_true"], y_p, labels=[0, 1, 2])
print("RAG Confusion Matrix (rows=actual, cols=predicted)\n")
print(pd.DataFrame(cm, index=labels, columns=labels))

RAG Confusion Matrix (rows=actual, cols=predicted)

           negative  positive  uncertain
negative         17         0          3
positive          1        16          3
uncertain         0         0          0


In [17]:
# Classification report
resolved = results_df["pred"].notna()
if resolved.sum() > 0:
    print(classification_report(
        results_df.loc[resolved, "y_true"],
        results_df.loc[resolved, "pred"].astype(int),
        target_names=["negative", "positive"],
    ))
else:
    print("No resolved predictions.")

              precision    recall  f1-score   support

    negative       0.94      1.00      0.97        17
    positive       1.00      0.94      0.97        17

    accuracy                           0.97        34
   macro avg       0.97      0.97      0.97        34
weighted avg       0.97      0.97      0.97        34



## Metric Interpretation

The RAG classified 40 test reports, making a definite verdict (true or false) on 34 and abstaining with "uncertain" on 6. Strict accuracy is 82.5% because uncertain counts as wrong; if we look only at resolved predictions, accuracy reaches 97.1%. The RAG is conservative: when unsure, it abstains rather than guessing. Precision is 100%—when it predicted pneumonia, it was always correct—and recall is 94.1%, so it missed a small share of true positives. The high F1 (0.97) reflects strong overall performance on definite predictions.